In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import json, os

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (BaggingClassifier, AdaBoostClassifier,
                              GradientBoostingClassifier, StackingClassifier)
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, roc_auc_score, roc_curve)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import seaborn as sns

PLOT_DIR = r'c:\ML\EXP7\plots'
os.makedirs(PLOT_DIR, exist_ok=True)

data = load_breast_cancer()
X, y = data.data, data.target
print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
# ── Bagging ───────────────────────────────────────────────────────────────────
bag_grid = [
    {'n_estimators': 10,  'max_samples': 0.8},
    {'n_estimators': 50,  'max_samples': 0.8},
    {'n_estimators': 100, 'max_samples': 1.0},
]
bag_results = []
for p in bag_grid:
    clf = BaggingClassifier(estimator=DecisionTreeClassifier(), random_state=42, n_jobs=1, **p)
    acc = cross_val_score(clf, X_train, y_train, cv=cv, scoring='accuracy')
    f1  = cross_val_score(clf, X_train, y_train, cv=cv, scoring='f1')
    bag_results.append({**p, 'avg_acc': round(acc.mean(),4), 'avg_f1': round(f1.mean(),4)})
    print(f"Bag {p} => acc={acc.mean():.4f} f1={f1.mean():.4f}")

best_bag = max(bag_results, key=lambda r: r['avg_acc'])
bag_clf = BaggingClassifier(estimator=DecisionTreeClassifier(),
    n_estimators=best_bag['n_estimators'], max_samples=best_bag['max_samples'],
    random_state=42, n_jobs=1)
bag_clf.fit(X_train, y_train)
y_pred_bag = bag_clf.predict(X_test)

In [ ]:
# ── AdaBoost ───────────────────────────────────────────────────────────────────
ada_grid = [
    {'n_estimators': 50,  'learning_rate': 0.5},
    {'n_estimators': 100, 'learning_rate': 1.0},
    {'n_estimators': 200, 'learning_rate': 0.1},
]
ada_results = []
for p in ada_grid:
    clf = AdaBoostClassifier(random_state=42, **p)
    acc = cross_val_score(clf, X_train, y_train, cv=cv, scoring='accuracy')
    f1  = cross_val_score(clf, X_train, y_train, cv=cv, scoring='f1')
    ada_results.append({**p, 'avg_acc': round(acc.mean(),4), 'avg_f1': round(f1.mean(),4)})
    print(f"Ada {p} => acc={acc.mean():.4f} f1={f1.mean():.4f}")

best_ada = max(ada_results, key=lambda r: r['avg_acc'])
ada_clf = AdaBoostClassifier(n_estimators=best_ada['n_estimators'],
    learning_rate=best_ada['learning_rate'], random_state=42)
ada_clf.fit(X_train, y_train)
y_pred_ada = ada_clf.predict(X_test)

In [ ]:
# ── GradientBoosting ──────────────────────────────────────────────────────────
gb_grid = [
    {'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 3},
    {'n_estimators': 200, 'learning_rate': 0.05,'max_depth': 3},
    {'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 5},
]
gb_results = []
for p in gb_grid:
    clf = GradientBoostingClassifier(random_state=42, **p)
    acc = cross_val_score(clf, X_train, y_train, cv=cv, scoring='accuracy')
    f1  = cross_val_score(clf, X_train, y_train, cv=cv, scoring='f1')
    gb_results.append({**p, 'avg_acc': round(acc.mean(),4), 'avg_f1': round(f1.mean(),4)})
    print(f"GB {p} => acc={acc.mean():.4f} f1={f1.mean():.4f}")

best_gb = max(gb_results, key=lambda r: r['avg_acc'])
gb_clf = GradientBoostingClassifier(n_estimators=best_gb['n_estimators'],
    learning_rate=best_gb['learning_rate'], max_depth=best_gb['max_depth'], random_state=42)
gb_clf.fit(X_train, y_train)
y_pred_gb = gb_clf.predict(X_test)

In [ ]:
# ── Stacking ───────────────────────────────────────────────────────────────────
estimators = [
    ('svm', Pipeline([('sc', StandardScaler()), ('svc', SVC(probability=True, kernel='rbf', C=1))])),
    ('nb',  GaussianNB()),
    ('dt',  DecisionTreeClassifier(max_depth=5, random_state=42)),
]
stack_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5, n_jobs=1)
stack_clf.fit(X_train, y_train)
y_pred_stack = stack_clf.predict(X_test)

stack_acc_cv = cross_val_score(stack_clf, X, y, cv=cv, scoring='accuracy')
stack_f1_cv  = cross_val_score(stack_clf, X, y, cv=cv, scoring='f1')
print(f"Stack CV acc={stack_acc_cv.mean():.4f} f1={stack_f1_cv.mean():.4f}")

In [ ]:
# ── Final Metrics ──────────────────────────────────────────────────────────────
def get_metrics(y_true, y_pred, clf=None, X_test=None):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred)
    if hasattr(clf, 'predict_proba') and X_test is not None:
        proba = clf.predict_proba(X_test)[:,1]
        auc = roc_auc_score(y_true, proba)
    else:
        auc = None
    return acc, prec, rec, f1, auc

m_bag   = get_metrics(y_test, y_pred_bag,   bag_clf,   X_test)
m_ada   = get_metrics(y_test, y_pred_ada,   ada_clf,   X_test)
m_gb    = get_metrics(y_test, y_pred_gb,    gb_clf,    X_test)
m_stack = get_metrics(y_test, y_pred_stack, stack_clf, X_test)

print("\n=== FINAL RESULTS ===")
for name, m in [('Bagging', m_bag), ('AdaBoost', m_ada), ('GradBoost', m_gb), ('Stacking', m_stack)]:
    print(f"{name}: Acc={m[0]:.4f} Prec={m[1]:.4f} Rec={m[2]:.4f} F1={m[3]:.4f} AUC={m[4]}")

In [ ]:
# ── Confusion Matrix Plot ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, y_pred, title in zip(axes,
        [y_pred_bag, y_pred_ada, y_pred_gb, y_pred_stack],
        ['Bagging', 'AdaBoost', 'GradBoost', 'Stacking']):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['M','B'], yticklabels=['M','B'])
    ax.set_title(title); ax.set_xlabel('Pred'); ax.set_ylabel('True')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'confusion_matrices.png'))
plt.savefig(os.path.join(PLOT_DIR, 'confusion_matrices.eps'))
plt.close()

In [ ]:
# ── ROC Curves ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7,6))
for clf, label, color in [
        (bag_clf, 'Bagging', '#e74c3c'), (ada_clf, 'AdaBoost', '#f39c12'),
        (gb_clf,  'GradBoost', '#2ecc71'), (stack_clf, 'Stacking', '#2980b9')]:
    proba = clf.predict_proba(X_test)[:,1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, label=f'{label} (AUC={auc:.3f})', color=color)
ax.plot([0,1],[0,1],'k--')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.set_title('ROC Curves'); ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'roc_curves.png'))
plt.savefig(os.path.join(PLOT_DIR, 'roc_curves.eps'))
plt.close()
print("All EXP7 plots saved.")

In [ ]:
# ── Save results JSON ─────────────────────────────────────────────────────────
results = {
    'bag_results': bag_results, 'ada_results': ada_results, 'gb_results': gb_results,
    'stack_cv_acc': round(stack_acc_cv.mean(),4), 'stack_cv_f1': round(stack_f1_cv.mean(),4),
    'best_bag': best_bag, 'best_ada': best_ada, 'best_gb': best_gb,
    'final': {
        'Bagging':   list(m_bag),
        'AdaBoost':  list(m_ada),
        'GradBoost': list(m_gb),
        'Stacking':  list(m_stack),
    }
}
with open(r'c:\ML\EXP7\results.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)
print("Results saved.")